# Build Overall Rank Table

This notebook creates **overall_rank.parquet**: a single table per work with popularity metrics and a combined **popularity_score** used for the **long tail of popularity** analysis.

## What we build
- **Input**: ratings_cleaned.parquet, reading_log_cleaned.parquet, works_cleaned.parquet (for title)
- **Output**: data/processed/overall_rank.parquet
- **Columns**: work_key, title, count_of_ratings, bayesian_rating, average_rating, want_to_read, already_read, currently_reading, norm_log_bayesian_rating, norm_log_already_read, norm_log_want_to_read, norm_log_currently_reading, popularity_score, overall_rank

## Popularity score formula (in text)
1. **Log-scale**: For each metric use log(1 + value) to squash high values.
2. **Normalize to 0–1**: Norm(x) = (x − min) / (max − min) across works (per metric).
3. **Weighted sum**: **popularity_score** = 1 × Norm(log(bayesian_rating)) + 0.8 × Norm(log(already_read)) + 0.4 × Norm(log(currently_reading)) + 0.1 × Norm(log(want_to_read))

Values are stored with 10 decimal places for detailed rank comparison.

## Step 1: Setup
Import libraries and set paths. Output goes to `data/processed/` and report to `reports/`.

In [13]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
PROCESSED_DIR = DATA_DIR / 'processed'
REPORTS_DIR = Path('../reports')
REPORTS_DIR.mkdir(exist_ok=True)

## Step 2: Load ratings (cleaned)
Load ratings_cleaned.parquet. We will aggregate **per work_key**: count of ratings and average rating.

In [14]:
ratings = pd.read_parquet(PROCESSED_DIR / 'ratings_cleaned.parquet')
print(f"Ratings rows: {len(ratings):,}")
print(ratings.head())

Ratings rows: 590,986
             work_key         edition_key  rating rating_date  rating_year
0  /works/OL17882343W                           3  2018-06-20         2018
1   /works/OL1629179W  /books/OL22981670M       5  2018-06-20         2018
2   /works/OL4226036W  /books/OL10690412M       5  2018-06-20         2018
3   /works/OL5264255W   /books/OL2719185M       5  2018-06-20         2018
4   /works/OL1681415W   /books/OL2582724M       5  2018-06-20         2018


## Step 3: Aggregate ratings per work
For each work_key compute:
- **count_of_ratings**: number of ratings
- **average_rating**: mean of rating (only non-null ratings)

In [15]:
ratings_agg = ratings.groupby('work_key').agg(
    count_of_ratings=('rating', 'count'),
    average_rating=('rating', 'mean')
).reset_index()

print(f"Works with at least one rating: {len(ratings_agg):,}")
print(ratings_agg.head(10))

Works with at least one rating: 274,266
             work_key  count_of_ratings  average_rating
0  /works/OL10000000W                 1        3.000000
1    /works/OL100001W                 1        1.000000
2   /works/OL1000035W                 1        4.000000
3   /works/OL1000043W                 1        3.000000
4    /works/OL100004W                 1        3.000000
5   /works/OL1000059W                 1        3.000000
6   /works/OL1000061W                 1        5.000000
7   /works/OL1000063W                 1        5.000000
8   /works/OL1000066W                 3        3.666667
9   /works/OL1000075W                 1        2.000000


## Step 4: Bayesian rating (formula in text)
**Bayesian rating** pulls the average toward a global prior when a work has few ratings.

Formula: **bayesian_rating** = (count × average_rating + m × global_mean) / (count + m)

- **m** = minimum vote weight (e.g. 10) — how strongly we pull toward the prior.
- **global_mean** = mean rating across all ratings.

Works with many ratings stay close to their average; works with few ratings move toward the global mean.

In [16]:
# Distribution of number of ratings per work (to choose m for Bayesian prior)
counts = ratings_agg['count_of_ratings']
print("Distribution of count_of_ratings (number of ratings per work)")
print("=" * 50)
print(f"  count (n works):     {len(ratings_agg):,}")
print(f"  min:                 {counts.min()}")
print(f"  max:                 {counts.max():,}")
print(f"  mean:                {counts.mean():.2f}")
print(f"  median:              {counts.median():.0f}")
print(f"  std:                 {counts.std():.2f}")
print()
print("Percentiles:")
for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]:
    print(f"  {p:3d}%: {counts.quantile(p/100):.0f}")
print()
# How many works have at most m ratings (for common m choices)
print("Works with count_of_ratings <= m (for choosing m):")
for m in [5, 10, 25, 50, 100, 200]:
    n = (counts <= m).sum()
    pct = 100 * n / len(ratings_agg)
    print(f"  m={m:3d}: {n:,} works ({pct:.1f}%)")

Distribution of count_of_ratings (number of ratings per work)
  count (n works):     274,266
  min:                 1
  max:                 1,275
  mean:                2.15
  median:              1
  std:                 9.37

Percentiles:
    1%: 1
    5%: 1
   10%: 1
   25%: 1
   50%: 1
   75%: 2
   90%: 3
   95%: 5
   99%: 17

Works with count_of_ratings <= m (for choosing m):
  m=  5: 260,753 works (95.1%)
  m= 10: 268,687 works (98.0%)
  m= 25: 272,708 works (99.4%)
  m= 50: 273,596 works (99.8%)
  m=100: 273,996 works (99.9%)
  m=200: 274,187 works (100.0%)


In [17]:
# m = 10: from distribution, 98% of works have ≤10 ratings; strong prior for long tail, little effect on popular works
MIN_VOTES = 10
global_mean = ratings['rating'].mean()
print(f"Global mean rating: {global_mean:.4f}")

ratings_agg['bayesian_rating'] = (
    (ratings_agg['count_of_ratings'] * ratings_agg['average_rating'] + MIN_VOTES * global_mean)
    / (ratings_agg['count_of_ratings'] + MIN_VOTES)
)
print(ratings_agg[['work_key', 'count_of_ratings', 'average_rating', 'bayesian_rating']].head(10))

Global mean rating: 3.9991
             work_key  count_of_ratings  average_rating  bayesian_rating
0  /works/OL10000000W                 1        3.000000         3.908317
1    /works/OL100001W                 1        1.000000         3.726499
2   /works/OL1000035W                 1        4.000000         3.999226
3   /works/OL1000043W                 1        3.000000         3.908317
4    /works/OL100004W                 1        3.000000         3.908317
5   /works/OL1000059W                 1        3.000000         3.908317
6   /works/OL1000061W                 1        5.000000         4.090135
7   /works/OL1000063W                 1        5.000000         4.090135
8   /works/OL1000066W                 3        3.666667         3.922422
9   /works/OL1000075W                 1        2.000000         3.817408


## Step 5: Load reading log (cleaned)
Load reading_log_cleaned.parquet. We will count per work_key how many **already_read**, **want_to_read**, and **currently_reading** entries exist.

In [18]:
log = pd.read_parquet(PROCESSED_DIR / 'reading_log_cleaned.parquet')
print(f"Reading log rows: {len(log):,}")
print("Unique status values:", log['status'].dropna().unique().tolist())
print(log.head())

Reading log rows: 11,529,486
Unique status values: ['already read', 'want to read', 'currently reading']
             work_key        edition_key        status    log_date  log_year
0   /works/OL4439701W                     already read  2017-12-11      2017
1     /works/OL63060W  /books/OL5816906M  already read  2017-12-26      2017
2  /works/OL10417330W                     want to read  2017-11-08      2017
3   /works/OL4466500W  /books/OL2712504M  want to read  2017-11-08      2017
4   /works/OL5920528W                     want to read  2018-01-05      2018


## Step 6: Count reading log status per work
For each work_key, count entries with status **already read**, **want to read**, and **currently reading** (case-insensitive match).

In [19]:
log['status_norm'] = log['status'].astype(str).str.strip().str.lower()

log_agg = log.groupby('work_key').agg(
    already_read=('status_norm', lambda s: (s == 'already read').sum()),
    want_to_read=('status_norm', lambda s: (s == 'want to read').sum()),
    currently_reading=('status_norm', lambda s: (s == 'currently reading').sum())
).reset_index()

print(f"Works in reading log: {len(log_agg):,}")
print(log_agg.head(10))

Works in reading log: 2,995,247
             work_key  already_read  want_to_read  currently_reading
0  /works/OL10000000W             1             0                  0
1  /works/OL10000001W             0             1                  0
2  /works/OL10000008W             0             1                  0
3   /works/OL1000001W             1             1                  0
4  /works/OL10000077W             0             1                  0
5   /works/OL1000008W             0             1                  0
6    /works/OL100000W             3             3                  0
7   /works/OL1000011W             1             1                  0
8  /works/OL10000143W             0             1                  0
9  /works/OL10000155W             0             2                  0


## Step 7: Merge ratings and reading log (all works)
Join ratings and reading-log aggregates on **work_key** (outer join). Fill missing counts with 0; fill missing average_rating with 0. For **bayesian_rating**: works with no ratings (count_of_ratings = 0) get 0; works that had ratings get their computed value (missing filled by global mean).

In [20]:
df = ratings_agg.merge(log_agg, on='work_key', how='outer')

df['count_of_ratings'] = df['count_of_ratings'].fillna(0).astype(int)
df['already_read'] = df['already_read'].fillna(0).astype(int)
df['want_to_read'] = df['want_to_read'].fillna(0).astype(int)
df['currently_reading'] = df['currently_reading'].fillna(0).astype(int)

df['average_rating'] = df['average_rating'].fillna(0.0)
df['bayesian_rating'] = df['bayesian_rating'].fillna(global_mean)
# Works with no ratings: set bayesian_rating to 0 (consistent with count=0, average=0)
df.loc[df['count_of_ratings'] == 0, 'bayesian_rating'] = 0.0

print(f"Total works: {len(df):,}")
print(df.head(10))
print(df.tail(10))

Total works: 3,000,008
             work_key  count_of_ratings  average_rating  bayesian_rating  \
0  /works/OL10000000W                 1             3.0         3.908317   
1  /works/OL10000001W                 0             0.0         0.000000   
2  /works/OL10000008W                 0             0.0         0.000000   
3   /works/OL1000001W                 0             0.0         0.000000   
4  /works/OL10000077W                 0             0.0         0.000000   
5   /works/OL1000008W                 0             0.0         0.000000   
6    /works/OL100000W                 0             0.0         0.000000   
7   /works/OL1000011W                 0             0.0         0.000000   
8  /works/OL10000143W                 0             0.0         0.000000   
9  /works/OL10000155W                 0             0.0         0.000000   

   already_read  want_to_read  currently_reading  
0             1             0                  0  
1             0             1         

## Step 7b: Add title from works_cleaned
Load works_cleaned.parquet and left-join **title** on **work_key** so each row shows the work title alongside work_key. Works not in works_cleaned keep title as null.

In [21]:
works = pd.read_parquet(PROCESSED_DIR / 'works_cleaned.parquet', columns=['work_key', 'title'])
df = df.merge(works[['work_key', 'title']], on='work_key', how='left')
print(f"Works with title: {df['title'].notna().sum():,} / {len(df):,}")
print(df[['work_key', 'title', 'count_of_ratings', 'bayesian_rating']].head(10))

Works with title: 2,966,900 / 3,000,008
             work_key                                              title  \
0  /works/OL10000000W             The catcher in the rye de j.d.salinger   
1  /works/OL10000001W                 Pride and prejudice de jane austen   
2  /works/OL10000008W               Les Fleurs bleues de Raymond Queneau   
3   /works/OL1000001W                                  Ghost Of A Chance   
4  /works/OL10000077W                               Classic garden style   
5   /works/OL1000008W                             My heart cries for you   
6    /works/OL100000W                                         Bloody Kin   
7   /works/OL1000011W                                      One dead dean   
8  /works/OL10000143W  Dhikrayat Isbaniyah wa-Andalusiyah maa Nizar Q...   
9  /works/OL10000155W  100 fiches de maths pour prépas commerciales a...   

   count_of_ratings  bayesian_rating  
0                 1         3.908317  
1                 0         0.000000  
2     

## Step 8: Log-scale, normalize to 0–1, then popularity score
1. **Log-scale**: For each metric compute log(1 + value) to squash high values.
2. **Normalize to 0–1**: Norm(x) = (x − min) / (max − min) per metric across works; store in norm_log_bayesian_rating, norm_log_already_read, norm_log_want_to_read, norm_log_currently_reading.
3. **Formula**: **popularity_score** = 1 × Norm(log(bayesian_rating)) + 0.8 × Norm(log(already_read)) + 0.4 × Norm(log(currently_reading)) + 0.1 × Norm(log(want_to_read))

We store the value with 10 decimal places for detailed rank comparison.

In [26]:
# Log-scale: log(1 + x) to squash high values and handle zeros
log_bayesian = np.log1p(df['bayesian_rating'])
log_already_read = np.log1p(df['already_read'])
log_want_to_read = np.log1p(df['want_to_read'])
log_currently_reading = np.log1p(df['currently_reading'])

# Normalize each to 0-1: (x - min) / (max - min); if max==min use 0
def norm_01(s):
    lo, hi = s.min(), s.max()
    if hi <= lo:
        return pd.Series(0.0, index=s.index)
    return (s - lo) / (hi - lo)

df['norm_log_bayesian_rating'] = norm_01(log_bayesian)
df['norm_log_already_read'] = norm_01(log_already_read)
df['norm_log_want_to_read'] = norm_01(log_want_to_read)
df['norm_log_currently_reading'] = norm_01(log_currently_reading)

# Popularity score: weighted sum of normalized log-scaled metrics
df['popularity_score'] = (
    1.0 * df['norm_log_bayesian_rating']
    + 1.0 * df['norm_log_already_read']
    + 0.9* df['norm_log_currently_reading']
    + 0.1 * df['norm_log_want_to_read']
)
df['popularity_score'] = np.round(df['popularity_score'], 10)

print(df[['work_key', 'norm_log_bayesian_rating', 'norm_log_already_read', 'norm_log_currently_reading', 'norm_log_want_to_read', 'popularity_score']].head(10))

             work_key  norm_log_bayesian_rating  norm_log_already_read  \
0  /works/OL17930368W                  0.912884               0.999911   
1  /works/OL18020194W                  0.937539               0.979628   
2     /works/OL82563W                  0.940572               1.000000   
3   /works/OL2010879W                  0.914776               0.966473   
4   /works/OL1968368W                  0.916286               0.952165   
5  /works/OL17590212W                  0.922422               0.900737   
6  /works/OL35351151W                  0.929791               0.882577   
7    /works/OL257943W                  0.938623               0.921127   
8  /works/OL25312237W                  0.892577               0.911227   
9     /works/OL82536W                  0.941813               0.952039   

   norm_log_currently_reading  norm_log_want_to_read  popularity_score  
0                    1.000000               1.000000          2.912795  
1                    0.936773          

## Step 9: Overall rank
**overall_rank** = rank of each work by **popularity_score** (descending). Rank 1 = highest score. We use pandas rank with method='min' so ties get the same rank.

In [27]:
df['overall_rank'] = df['popularity_score'].rank(ascending=False, method='min').astype(int)
df = df.sort_values('overall_rank').reset_index(drop=True)

cols = [
    'work_key', 'title', 'count_of_ratings', 'bayesian_rating', 'average_rating',
    'want_to_read', 'already_read', 'currently_reading',
    'norm_log_bayesian_rating', 'norm_log_already_read', 'norm_log_want_to_read', 'norm_log_currently_reading',
    'popularity_score', 'overall_rank'
]
df = df[cols]

print(df.head(10))

             work_key                                       title  \
0  /works/OL17930368W                               Atomic Habits   
1  /works/OL18020194W                             It Ends With Us   
2   /works/OL2010879W                          Rich Dad, Poor Dad   
3   /works/OL1968368W                        The 48 Laws of Power   
4     /works/OL82563W    Harry Potter and the Philosopher's Stone   
5  /works/OL17590212W         The Subtle Art of Not Giving a F*ck   
6  /works/OL35351151W                      Um casamento arranjado   
7  /works/OL25312237W  Control Your Mind and Master Your Feelings   
8    /works/OL257943W                           A Game of Thrones   
9    /works/OL527464W                         Think and Grow Rich   

   count_of_ratings  bayesian_rating  average_rating  want_to_read  \
0              1275         3.985207        3.985098         51271   
1              1042         4.206266        4.208253         39757   
2              1077         4.

## Step 10: Save table
Save the final table to **data/processed/overall_rank.parquet**.

In [28]:
out_path = PROCESSED_DIR / 'overall_rank.parquet'
df.to_parquet(out_path, index=False)
print(f"Saved to {out_path}")
print(f"Rows: {len(df):,}")
print(f"File size: {out_path.stat().st_size / 1024**2:.2f} MB")

Saved to ../data/processed/overall_rank.parquet
Rows: 3,000,008
File size: 89.81 MB


## Step 11: Combined report
Write a short report to **reports/overall_rank_report.md** with: formula, column descriptions, and summary stats.

In [29]:
report = f"""# Overall Rank Table — Report

## Purpose
This table feeds the **long tail of popularity** analysis. Each row is one work (book) with a combined **popularity_score** and **overall_rank**.

## Output file
- **Path**: data/processed/overall_rank.parquet
- **Rows**: {len(df):,}

## Columns
| Column | Description |
|--------|-------------|
| work_key | Unique work identifier (e.g. /works/OL123W) |
| title | Work title from works_cleaned (null if not in works) |
| count_of_ratings | Number of ratings for this work |
| bayesian_rating | Bayesian average rating (prior-pulled when few ratings) |
| average_rating | Simple average rating |
| want_to_read | Count of "want to read" in reading log |
| already_read | Count of "already read" in reading log |
| currently_reading | Count of "currently reading" in reading log |
| norm_log_bayesian_rating | Norm(log(1 + bayesian_rating)) in [0, 1] |
| norm_log_already_read | Norm(log(1 + already_read)) in [0, 1] |
| norm_log_want_to_read | Norm(log(1 + want_to_read)) in [0, 1] |
| norm_log_currently_reading | Norm(log(1 + currently_reading)) in [0, 1] |
| popularity_score | Combined score (see formula) |
| overall_rank | Rank by popularity_score (1 = highest) |

## Popularity score formula
1. Log-scale: log(1 + value) per metric. 2. Normalize to 0–1: Norm(x) = (x − min) / (max − min) per metric.  
**popularity_score** = 1 × Norm(log(bayesian_rating)) + 0.8 × Norm(log(already_read)) + 0.4 × Norm(log(currently_reading)) + 0.1 × Norm(log(want_to_read))

Values are stored with 10 decimal places for fine rank comparison.

## Summary statistics
- popularity_score — min: {df['popularity_score'].min():.10f}, max: {df['popularity_score'].max():.10f}, mean: {df['popularity_score'].mean():.10f}
- overall_rank — 1 to {df['overall_rank'].max():,}
- count_of_ratings — max: {df['count_of_ratings'].max():,}
- already_read — max: {df['already_read'].max():,}
"""

report_path = REPORTS_DIR / 'overall_rank_report.md'
report_path.write_text(report)
print(f"Report saved to {report_path}")
print(report)

Report saved to ../reports/overall_rank_report.md
# Overall Rank Table — Report

## Purpose
This table feeds the **long tail of popularity** analysis. Each row is one work (book) with a combined **popularity_score** and **overall_rank**.

## Output file
- **Path**: data/processed/overall_rank.parquet
- **Rows**: 3,000,008

## Columns
| Column | Description |
|--------|-------------|
| work_key | Unique work identifier (e.g. /works/OL123W) |
| title | Work title from works_cleaned (null if not in works) |
| count_of_ratings | Number of ratings for this work |
| bayesian_rating | Bayesian average rating (prior-pulled when few ratings) |
| average_rating | Simple average rating |
| want_to_read | Count of "want to read" in reading log |
| already_read | Count of "already read" in reading log |
| currently_reading | Count of "currently reading" in reading log |
| norm_log_bayesian_rating | Norm(log(1 + bayesian_rating)) in [0, 1] |
| norm_log_already_read | Norm(log(1 + already_read)) in [